### Ant Colony Optimization (ACO) for Sudoku

Now, let's define the core functions for the Ant Colony Optimization algorithm: `is_valid_placement`, `initialize_pheromones`, `calculate_desirability`, `ant_construct_solution`, `update_pheromones`, and `ant_colony_optimization`.

### Sudoku Boards Definitions

In [ ]:
sudoku_4x4 = [
    [1, 2, 0, 0],
    [3, 4, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 3, 4]
]

sudoku_9x9 = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

In [ ]:
import numpy as np
import pandas as pd
import random
from copy import deepcopy
from tqdm.notebook import trange

def fitness(board, N):
    conflicts = 0
    sqrt_N = int(np.sqrt(N))

    for r in range(N):
        row_values = board[r, :]
        conflicts += len(row_values) - len(np.unique(row_values[row_values != 0]))

    for c in range(N):
        col_values = board[:, c]
        conflicts += len(col_values) - len(np.unique(col_values[col_values != 0]))

    for r_block in range(sqrt_N):
        for c_block in range(sqrt_N):
            block_values = board[r_block * sqrt_N:(r_block + 1) * sqrt_N, c_block * sqrt_N:(c_block + 1) * sqrt_N].flatten()
            conflicts += len(block_values) - len(np.unique(block_values[block_values != 0]))
    return conflicts

def get_fixed_positions(board, N):
    fixed = []
    for r in range(N):
        for c in range(N):
            if board[r, c] != 0:
                fixed.append((r, c))
    return fixed

# Parameters for ACO 4x4 Sudoku
num_ants_aco_4x4 = [10, 20]
evaporation_rates_aco_4x4 = [0.1, 0.3]
Q_values_aco_4x4 = [0.5, 1.0]
max_generations_aco_4x4 = 500

# Parameters for ACO 9x9 Sudoku
num_ants_aco_9x9 = [20, 50]
evaporation_rates_aco_9x9 = [0.1, 0.3]
Q_values_aco_9x9 = [0.5, 1.0]
max_generations_aco_9x9 = 1000

repeats_aco = 1

In [ ]:
def is_valid_placement(board, row, col, num, N):
    if num in board[row, :]:
        return False
    if num in board[:, col]:
        return False
    sqrt_N = int(np.sqrt(N))
    start_row, start_col = sqrt_N * (row // sqrt_N), sqrt_N * (col // sqrt_N)
    if num in board[start_row:start_row + sqrt_N, start_col:start_col + sqrt_N]:
        return False
    return True

def initialize_pheromones(N, initial_pheromone=0.1):
    return np.full((N, N, N + 1), initial_pheromone, dtype=float)

def calculate_desirability(pheromones, r, c, num, N, alpha=1.0, beta=1.0):
    phi = pheromones[r, c, num]**alpha
    return phi

def ant_construct_solution(board_template, fixed_pos, pheromones, N):
    current_board = deepcopy(board_template)
    empty_cells = [(r, c) for r in range(N) for c in range(N) if board_template[r][c] == 0]
    random.shuffle(empty_cells)

    choices = []

    for r, c in empty_cells:
        possible_values = []
        probabilities = []

        for num in range(1, N + 1):
            if is_valid_placement(current_board, r, c, num, N):
                possible_values.append(num)
                probabilities.append(pheromones[r, c, num])

        if not possible_values:
            current_board[r, c] = random.randint(1, N)
            choices.append((r, c, current_board[r, c]))
            continue

        total_prob = sum(probabilities)
        if total_prob == 0:
            chosen_num = random.choice(possible_values)
        else:
            probabilities = [p / total_prob for p in probabilities]
            chosen_num = random.choices(possible_values, weights=probabilities, k=1)[0]

        current_board[r, c] = chosen_num
        choices.append((r, c, chosen_num))

    return current_board, choices

def update_pheromones(pheromones, best_solution, initial_board, N, evaporation_rate, Q):
    pheromones *= (1 - evaporation_rate)
    best_fitness = fitness(best_solution, N)

    if best_fitness == 0:
        delta_pheromone = Q
    else:
        delta_pheromone = Q / (best_fitness + 1e-6)

    for r in range(N):
        for c in range(N):
            if initial_board[r][c] == 0:
                chosen_num = best_solution[r, c]
                if chosen_num != 0:
                    pheromones[r, c, chosen_num] += delta_pheromone

def ant_colony_optimization(board, N, generations, num_ants, evaporation_rate, Q, initial_pheromone=0.1):
    pheromones = initialize_pheromones(N, initial_pheromone)
    best_overall_solution = None
    best_overall_fitness = float('inf')
    fixed_pos = get_fixed_positions(board, N)

    for gen in trange(generations, desc=f"ACO N={N}, Ants={num_ants}, Evap={evaporation_rate}", leave=False):
        current_best_ant_solution = None
        current_best_ant_fitness = float('inf')

        for _ in range(num_ants):
            ant_solution, ant_choices = ant_construct_solution(board, fixed_pos, pheromones, N)
            ant_fitness = fitness(ant_solution, N)

            if ant_fitness < current_best_ant_fitness:
                current_best_ant_fitness = ant_fitness
                current_best_ant_solution = ant_solution

        if current_best_ant_fitness < best_overall_fitness:
            best_overall_fitness = current_best_ant_fitness
            best_overall_solution = current_best_ant_solution
            if best_overall_fitness == 0:
                print(f"Solved by ACO in generation {gen}!")
                return True, gen, best_overall_solution

        if best_overall_solution is not None:
            update_pheromones(pheromones, best_overall_solution, board, N, evaporation_rate, Q)

    return False, generations, best_overall_solution

In [ ]:
def run_experiment_aco(sudoku_board, N, num_ants_list, evaporation_rates, Q_values, repeats=1, max_generations=1000):
    results = []

    for num_ants in num_ants_list:
        for evap_rate in evaporation_rates:
            for Q_val in Q_values:
                success_count = 0
                total_generations_for_solved = 0
                best_overall_board = None
                best_overall_fitness = float('inf')

                for r in trange(repeats, desc=f"ACO N={N}, Ants={num_ants}, Evap={evap_rate}, Q={Q_val}", leave=False):
                    solved, gens, final_board = ant_colony_optimization(
                        board=sudoku_board, N=N, generations=max_generations,
                        num_ants=num_ants, evaporation_rate=evap_rate, Q=Q_val
                    )

                    if solved:
                        success_count += 1
                        total_generations_for_solved += (gens + 1)
                        print(f"Solved N={N} with Ants={num_ants}, Evap={evap_rate}, Q={Q_val} in {gens} generations (Repeat {r+1}/{repeats}):")
                        print(final_board)
                        if best_overall_board is None or fitness(final_board, N) < best_overall_fitness:
                            best_overall_board = final_board
                            best_overall_fitness = fitness(final_board, N)
                    elif final_board is not None:
                        current_run_fitness = fitness(final_board, N)
                        if current_run_fitness < best_overall_fitness:
                            best_overall_fitness = current_run_fitness
                            best_overall_board = final_board

                avg_generations_solved = round(total_generations_for_solved / success_count, 2) if success_count > 0 else np.nan

                results.append({
                    "Lentelės dydis": f"{N}x{N}",
                    "Skruzdžių skaičius": num_ants,
                    "Garavimo greitis": evap_rate,
                    "Pheromone Q": Q_val,
                    "Sėkmės dažnis": f"{success_count}/{repeats}",
                    "Vid. generacijos (sėkm.)": avg_generations_solved
                })

                if success_count == 0 and best_overall_board is not None:
                    print(f"For N={N}, Ants={num_ants}, Evap={evap_rate}, Q={Q_val}: No solution found in any repeat. Best board found (fitness {best_overall_fitness}):\n")
                    print(best_overall_board)
    return pd.DataFrame(results)

In [ ]:
# Convert boards to numpy arrays for processing
sudoku_4x4_array = np.array(sudoku_4x4)
sudoku_9x9_array = np.array(sudoku_9x9)

# Run ACO experiments for 4x4
print("Running ACO experiments for 4x4 Sudoku...")
results_4x4 = run_experiment_aco(
    sudoku_4x4_array, 4, num_ants_aco_4x4, evaporation_rates_aco_4x4, 
    Q_values_aco_4x4, repeats=repeats_aco, max_generations=max_generations_aco_4x4
)
print(results_4x4)

# Run ACO experiments for 9x9
print("\nRunning ACO experiments for 9x9 Sudoku...")
results_9x9 = run_experiment_aco(
    sudoku_9x9_array, 9, num_ants_aco_9x9, evaporation_rates_aco_9x9, 
    Q_values_aco_9x9, repeats=repeats_aco, max_generations=max_generations_aco_9x9
)
print(results_9x9)